# Laboratorio 2: Consolidacion de comprobantes Sofland

Leeremos varios Excel, conservaremos el origen, los consolidaremos y visualizaremos sus totales. Usaremos datos sinteticos.

## Objetivos

- Leer archivos con pandas.
- Detectar el encabezado real.
- Concatenar empresas con trazabilidad.
- Resumir y visualizar Debe/Haber.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'laboratorios' else Path.cwd()
EJEMPLOS = ROOT / 'ejemplos'
print(EJEMPLOS.resolve())

## 1. Inspeccionar el Excel

Sofland agrega cuatro filas antes del encabezado real. Observamos el archivo sin modificarlo.

In [ ]:
archivo = EJEMPLOS / 'empresaa.xlsx'
crudo = pd.read_excel(archivo, header=None, nrows=10)
crudo

## 2. Detectar y limpiar el encabezado

Buscamos Cuenta, Debe y Haber para no depender ciegamente de skiprows=4.

In [ ]:
def detectar_fila_encabezado(path, max_rows=12):
    preview = pd.read_excel(path, header=None, nrows=max_rows)
    requeridas = {'Cuenta', 'Debe', 'Haber'}
    for indice, fila in preview.iterrows():
        valores = {str(valor).strip() for valor in fila.dropna()}
        if requeridas.issubset(valores):
            return int(indice)
    raise ValueError(f'No se encontro encabezado en {path}')

fila = detectar_fila_encabezado(archivo)
limpio = pd.read_excel(archivo, skiprows=fila).dropna(how='all')
print(f'Fila detectada: {fila}')
limpio.head()

## 3. Consolidar empresas

Agregamos una columna de trazabilidad antes de usar concat.

In [ ]:
def leer_empresa(path):
    encabezado = detectar_fila_encabezado(path)
    frame = pd.read_excel(path, skiprows=encabezado).dropna(how='all')
    frame['_empresa_origen'] = path.stem
    return frame

archivos = sorted(EJEMPLOS.glob('empresa*.xlsx'))
tablas = [leer_empresa(path) for path in archivos]
consolidado = pd.concat(tablas, ignore_index=True)
print(f'Archivos: {len(archivos)} | Filas: {len(consolidado)}')
consolidado.head()

## 4. Resumir y visualizar

El grafico ayuda a explorar; no reemplaza la aprobacion contable.

In [ ]:
resumen = (consolidado.groupby('_empresa_origen', as_index=False)
    .agg(total_debe=('Debe', 'sum'), total_haber=('Haber', 'sum')))
resumen['diferencia'] = resumen['total_debe'] - resumen['total_haber']
resumen

In [ ]:
ax = resumen.set_index('_empresa_origen')[['total_debe', 'total_haber']].plot(
    kind='bar', figsize=(10, 5), title='Debe y Haber por empresa')
ax.set_ylabel('Monto')
ax.set_xlabel('Empresa')
plt.tight_layout()
plt.show()

## Ejercicio de cierre

1. Filtra EmpresaE y explica por que su diferencia no es cero.
2. Crea un grafico por Cuenta.
3. Anota que hallazgos requieren decision humana antes de importar a Sofland.